# 02 — Model A: transfer-learning baseline

Train MobileNetV2 on original training images transformed to 128×128. A stratified validation subset is derived only from the persisted training manifest. The reserved test manifest is intentionally not loaded in this notebook. Model selection uses validation F1.

## Local and Colab setup

In Colab, this notebook mounts Google Drive, clones the public repository when needed, and installs it in editable mode. Upload the raw dataset to `MyDrive/Applied-AI-Midterm/data/raw/` with one folder per class. Checkpoints are written to Drive so training can resume after a runtime reset.

In [ ]:
import importlib
import subprocess
import sys
from pathlib import Path

IN_COLAB = importlib.util.find_spec('google.colab') is not None
if IN_COLAB:
    drive = importlib.import_module('google.colab').drive
    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/MLawson-Applied-AI-Midterm')
    if not PROJECT_ROOT.is_dir():
        subprocess.run(
            [
                'git',
                'clone',
                'https://github.com/mlaw1123/MLawson-Applied-AI-Midterm.git',
                str(PROJECT_ROOT),
            ],
            check=True,
        )
    subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-e', str(PROJECT_ROOT)],
        check=True,
    )
else:
    PROJECT_ROOT = Path.cwd()
    if PROJECT_ROOT.name == 'notebooks':
        PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT / 'src'))
print(f'Project root: {PROJECT_ROOT}')

In [ ]:
from dataclasses import asdict

import matplotlib.pyplot as plt
from torch import nn
from torch.optim import AdamW
from torch.optim.lr_scheduler import StepLR

from applied_ai_midterm.classifier import create_binary_mobilenet
from applied_ai_midterm.config import load_config
from applied_ai_midterm.datasets import create_classifier_dataloaders
from applied_ai_midterm.training import (
    fit_classifier,
    seed_everything,
    select_device,
)

## Reproducible paths and hyperparameters

Only `data/splits/train.csv` is supplied to the loader. The validation subset is stratified with the project seed and cannot contain reserved test records.

In [ ]:
config = load_config(PROJECT_ROOT / 'configs' / 'config.yaml')
seed_everything(config.random_seed)
device = select_device()

if IN_COLAB:
    DRIVE_ROOT = Path('/content/drive/MyDrive/Applied-AI-Midterm')
    RAW_DIR = DRIVE_ROOT / 'data' / 'raw'
    CHECKPOINT_DIR = DRIVE_ROOT / 'artifacts' / 'checkpoints' / 'model_a'
else:
    RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
    CHECKPOINT_DIR = PROJECT_ROOT / 'artifacts' / 'checkpoints' / 'model_a'

TRAIN_MANIFEST = PROJECT_ROOT / 'data' / 'splits' / 'train.csv'
VALIDATION_RATIO = 0.20
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4

print(f'Device: {device}')
print(f'Raw data: {RAW_DIR}')
print(f'Checkpoints: {CHECKPOINT_DIR}')

In [ ]:
loaders = create_classifier_dataloaders(
    TRAIN_MANIFEST,
    RAW_DIR,
    image_size=config.high_resolution_size,
    batch_size=config.classifier_batch_size,
    validation_ratio=VALIDATION_RATIO,
    random_seed=config.random_seed,
    num_workers=config.num_workers,
)
print(f'Training examples: {loaders.train_size:,}')
print(f'Validation examples: {loaders.validation_size:,}')
print(f'Class mapping: {loaders.class_mapping}')

## Initialize or resume Model A

`BCEWithLogitsLoss` consumes the model's single raw logit. Sigmoid is used only inside metric calculation. The latest checkpoint resumes all model, optimizer, scheduler, history, configuration, seed, class-mapping, and random-number-generator state.

In [ ]:
model = create_binary_mobilenet(pretrained=True)
criterion = nn.BCEWithLogitsLoss()
optimizer = AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)
scheduler = StepLR(optimizer, step_size=7, gamma=0.1)
latest_checkpoint = CHECKPOINT_DIR / 'classifier_a_latest.pt'
resume_from = latest_checkpoint if latest_checkpoint.is_file() else None
resume_description = str(resume_from) if resume_from else 'none — starting epoch 1'
print(f'Resume checkpoint: {resume_description}')

In [ ]:
run_configuration = {
    **asdict(config),
    'architecture': 'mobilenet_v2',
    'pretrained_weights': 'MobileNet_V2_Weights.DEFAULT',
    'validation_ratio': VALIDATION_RATIO,
    'learning_rate': LEARNING_RATE,
    'weight_decay': WEIGHT_DECAY,
    'scheduler': 'StepLR(step_size=7, gamma=0.1)',
}
history = fit_classifier(
    model,
    loaders.train,
    loaders.validation,
    optimizer,
    criterion,
    epochs=config.classifier_epochs,
    device=device,
    checkpoint_dir=CHECKPOINT_DIR,
    class_mapping=loaders.class_mapping,
    random_seed=config.random_seed,
    configuration=run_configuration,
    scheduler=scheduler,
    resume_from=resume_from,
)

## Training and validation history

These are development curves only. Final reserved-test evaluation belongs in the comparison phase after both classifiers are trained.

In [ ]:
train_history = history['train']
validation_history = history['validation']
epochs = [int(row['epoch']) for row in train_history]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for split_name, rows in (
    ('Train', train_history),
    ('Validation', validation_history),
):
    axes[0].plot(epochs, [row['loss'] for row in rows], label=split_name)
    axes[1].plot(epochs, [row['f1'] for row in rows], label=split_name)
    axes[2].plot(epochs, [row['roc_auc'] for row in rows], label=split_name)
for axis, title in zip(axes, ('Loss', 'F1', 'ROC AUC'), strict=True):
    axis.set_title(title)
    axis.set_xlabel('Epoch')
    axis.legend()
    axis.grid(alpha=0.25)
plt.suptitle('Model A development history')
plt.tight_layout()

print('Final validation metrics:')
validation_history[-1]